# Prompt → Policy: HalfCheetah RL with LLM-generated rewards

Companion notebook for the *Finding Theta* blog post. The pipeline:

1. **Anthropic's Claude** turns a natural-language description into a JSON reward spec.
2. **PPO** trains a HalfCheetah policy against that reward for a fixed step budget.
3. We render a deterministic rollout video of the trained policy.

Two paths through this notebook:
- **Quick test** (no API key needed): a short training with a hand-written reward to confirm everything works.
- **Full end-to-end** (Anthropic API key required): natural-language prompt → LLM → 200k-step training → video.

Recommended runtime: T4 GPU (Runtime → Change runtime type → T4).

## 1. Install and set up

Installs the package from GitHub. Set `BRANCH` if you want to pin a feature branch or a tagged release.

If you want the full end-to-end path, add your Anthropic API key as a Colab secret named `ANTHROPIC_API_KEY` (Tools → Secrets → New secret). Without it, the LLM cell will skip itself and you can still run the quick test.

In [ ]:
BRANCH = "main"  # replace with a branch/tag/commit pin if needed

!pip install --quiet "git+https://github.com/kuds/rl-llm-reward.git@{BRANCH}"

In [ ]:
import os

# MuJoCo needs an OpenGL backend for rgb_array rendering. Colab supports EGL out of the box.
os.environ.setdefault("MUJOCO_GL", "egl")

# Try to load the API key from Colab secrets, then fall back to env.
_in_colab = False
try:
    from google.colab import userdata
    _in_colab = True
    try:
        key = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        key = None
    if key:
        os.environ["ANTHROPIC_API_KEY"] = key
except ImportError:
    pass

if os.environ.get("ANTHROPIC_API_KEY"):
    print("ANTHROPIC_API_KEY: set. Full end-to-end path is enabled.")
else:
    msg = "Add ANTHROPIC_API_KEY as a Colab secret" if _in_colab else "export ANTHROPIC_API_KEY=..."
    print(f"ANTHROPIC_API_KEY: not set. The full end-to-end path will skip itself ({msg}).")

## 2. Quick test (no API key required)

A short PPO run with a hand-written reward (forward velocity + tiny control-cost penalty). Verifies the env, training harness, and video rendering all work. Around 2–3 minutes on a T4, longer on CPU.

10k steps is well below the 200k budget needed for a *good* policy — expect a wobbly, slow gait, not a fast run. The point is the pipeline, not the result.

In [ ]:
from pathlib import Path

from prompt_to_policy.reward import RewardSpec
from prompt_to_policy.train import HalfCheetahPPOConfig, train

quick_spec = RewardSpec.model_validate({
    "components": [
        {"feature": "forward_velocity", "weight": 1.0},
        {"feature": "control_cost", "weight": -0.05},
    ],
})

quick_result = train(
    spec=quick_spec,
    prompt="(quick test, hand-written) run forward",
    config=HalfCheetahPPOConfig(eval_episodes=2, video_length_steps=300),
    timesteps=10_000,
    run_dir=Path("runs/colab_quick"),
    seed=0,
    record_video=True,
)

print(f"final mean return : {quick_result.final_mean_return:.1f}")
print(f"mean ep length    : {quick_result.final_mean_length:.0f} (truncates at 1000)")
print(f"train seconds     : {quick_result.train_seconds:.1f}")
print(f"video             : {quick_result.video_path}")

In [ ]:
from IPython.display import Video

Video(str(quick_result.video_path), embed=True, width=480)

## 3. Full end-to-end (API key required)

Edit `PROMPT` to ask for any behavior the feature registry can express — some that work well: `"run backward"`, `"stand still and stay upright"`, `"move as little as possible while staying level"`. The LLM converts the prompt into a reward spec; PPO trains for `TIMESTEPS` steps; we render the resulting policy.

200k steps is the recommended quick budget. For better gaits push to 1M (`HalfCheetahPPOConfig.total_timesteps`). On a T4 this is roughly 5–10 min for 200k, 25–40 min for 1M.

If `ANTHROPIC_API_KEY` is not set, this cell is a no-op.

In [ ]:
PROMPT = "make the cheetah run forward as fast as possible"
TIMESTEPS = 200_000

if not os.environ.get("ANTHROPIC_API_KEY"):
    print("Skipping: ANTHROPIC_API_KEY is not set. Add it via Colab secrets and re-run.")
    e2e_result = None
else:
    from prompt_to_policy.envs import halfcheetah as hc
    from prompt_to_policy.llm import LLMRewardClient

    client = LLMRewardClient(
        feature_docs=hc.FEATURE_DOCS,
        cache_dir=Path("colab_llm_cache"),
    )
    gen = client.generate(PROMPT)

    print(f"prompt    : {PROMPT!r}")
    print(f"cached    : {gen.cached}")
    print(f"cost_usd  : {gen.estimated_cost_usd:.4f}")
    print(f"spec      : {gen.spec.model_dump_json()}")
    print()

    e2e_result = train(
        spec=gen.spec,
        prompt=PROMPT,
        timesteps=TIMESTEPS,
        run_dir=Path("runs/colab_e2e"),
        seed=0,
        record_video=True,
    )
    print(f"final mean return : {e2e_result.final_mean_return:.1f}")
    print(f"mean ep length    : {e2e_result.final_mean_length:.0f}")
    print(f"train seconds     : {e2e_result.train_seconds:.1f}")
    print(f"video             : {e2e_result.video_path}")

In [ ]:
if e2e_result is not None and e2e_result.video_path is not None:
    Video(str(e2e_result.video_path), embed=True, width=480)
else:
    print("No video to display (LLM step was skipped or rendering failed).")

## Where to go from here

- Edit `PROMPT` and re-run cell 3. The LLM cache means re-running the same prompt is free.
- Compare what happens when the LLM produces a reward whose intent matches the prompt vs. one that doesn't — that's the *good* failure mode the post is about.
- Run with `TIMESTEPS = 1_000_000` for full-quality policies. Plan for ~30 min on a T4.
- The full feature registry, prompt template, and training hyperparameters live in `src/prompt_to_policy/`. Hyperparameters are fixed; only the reward changes per prompt.